# Expense Review Agent -- Development Notebook

**Business problem.** Accounts Payable teams must review every submitted employee expense claim for policy compliance (spending limits, missing receipts, segregation-of-duties/self-approval, duplicate or "structured" split submissions) before reimbursement. Doing this manually for every transaction does not scale and is inconsistent between reviewers. A first-pass AI triage agent can auto-clear low-risk claims and route only the risky ones to a human reviewer, cutting review workload while improving audit consistency.

**This notebook** develops and evaluates the same agent that is deployed in `app.py` (Streamlit) and packaged in `agent.py`. It:
1. Generates/loads a simulated expense dataset (fully synthetic -- safe for public use)
2. Explores the data (EDA)
3. Implements rule-based policy checks (explainable, deterministic)
4. Trains an Isolation Forest anomaly detector on transaction features
5. Blends both signals into a 0-100 risk score and Low/Medium/High tier
6. Evaluates results against the known, intentionally-injected anomalies
7. Produces the same explanations and recommendations shown in the deployed app

> To run this notebook standalone in Google Colab, either upload `agent.py` and `data/sample_expenses.csv` from the GitHub repo to the Colab file browser, or run `!git clone <YOUR_REPO_URL>` in the first cell below and `%cd` into it.

In [ ]:
# Option A: clone the project repo (recommended -- replace with your repo URL)
# !git clone https://github.com/<your-username>/expense-review-agent.git
# %cd expense-review-agent

# Option B: if you've already uploaded agent.py + data/sample_expenses.csv to this
# Colab session, just continue to the next cell.

!pip install -q scikit-learn pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)

df = pd.read_csv('data/sample_expenses.csv')
print(df.shape)
df.head()

## 1. Exploratory Data Analysis

In [ ]:
df.info()
print()
print(df.describe(numeric_only=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['amount'].hist(bins=40, ax=axes[0])
axes[0].set_title('Distribution of claim amounts')
axes[0].set_xlabel('Amount ($)')

df['category'].value_counts().plot(kind='barh', ax=axes[1])
axes[1].set_title('Transactions by category')
plt.tight_layout()
plt.show()

In [ ]:
print('Missing receipts:', (~df['receipt_attached']).sum(), 'of', len(df))
print('Self-approved (approver == claimant):',
      (df['approved_by'].str.lower() == df['employee_name'].str.lower()).sum())
df['date'] = pd.to_datetime(df['date'])
print('Weekend submissions:', (df['date'].dt.dayofweek >= 5).sum())

## 2. Rule-based policy checks

These mirror standard AP/audit controls: category spending limits, mandatory receipts above a
threshold, segregation-of-duties (no self-approval), suspiciously round amounts, weekend
submissions, and duplicate/"structured" split transactions. Rules are deterministic and fully
explainable -- required for anything that will affect a real reimbursement decision.

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import the production agent.py from the repo root
from agent import ExpenseReviewAgent, AgentConfig, DEFAULT_CATEGORY_LIMITS

print(DEFAULT_CATEGORY_LIMITS)

## 3. Run the full agent pipeline

`agent.run()` applies the rule checks, fits an `IsolationForest` on encoded transaction
features (amount, day-of-week, category, department, vendor, payment method), blends the two
signals into a risk score, and generates plain-language explanations -- exactly what the
deployed Streamlit app does.

In [ ]:
agent = ExpenseReviewAgent(AgentConfig(contamination=0.10))
scored = agent.run(df)
scored[['transaction_id', 'employee_name', 'category', 'amount', 'risk_score', 'risk_tier', 'recommended_action']].sort_values('risk_score', ascending=False).head(15)

## 4. Evaluation

The sample dataset was generated with `generate_sample_data.py`, which intentionally injects
known policy violations (over-limit claims, missing receipts, round-number amounts,
self-approvals, weekend submissions, and split/structured transactions) at roughly an 8% rate,
similar to typical real-world exception rates. We check that the agent's High/Medium tiers
capture the injected anomalies at a much higher rate than the general population -- a simple
precision/recall-style sanity check for an unsupervised/rule-hybrid system where we don't have
ground-truth labels on every real-world row (a supervised metric isn't available in the
one-off synthetic set, so we instead validate flag coverage per injected anomaly type).

In [ ]:
stats = agent.summary_stats(scored)
for k, v in stats.items():
    print(f'{k}: {v}')

print()
flag_text = scored['rule_flags'].apply(lambda fl: ' '.join(fl))
coverage = {
    'over_limit ("Exceeds")': flag_text.str.contains('Exceeds').mean(),
    'missing_receipt': flag_text.str.contains('No receipt').mean(),
    'round_number': flag_text.str.contains('round amount').mean(),
    'self_approved': flag_text.str.contains('Self-approved').mean(),
    'weekend': flag_text.str.contains('weekend').mean(),
    'duplicate_or_structuring': flag_text.str.contains('duplicate|structured', regex=True).mean(),
}
print('Share of ALL transactions carrying each flag type (dataset ~8% intentionally anomalous):')
for k, v in coverage.items():
    print(f'  {k}: {v:.1%}')

In [ ]:
scored['risk_tier'].value_counts().plot(kind='bar', title='Risk tier distribution')
plt.ylabel('# transactions')
plt.show()

## 5. Example agent explanation for a flagged transaction

This is the same explanation text shown in the Streamlit app's transaction detail view --
important for auditability: a reviewer should never have to guess *why* the agent flagged
something.

In [ ]:
example = scored.sort_values('risk_score', ascending=False).iloc[0]
print('Transaction:', example['transaction_id'])
print('Employee:', example['employee_name'], '|', example['category'], '|', f"${example['amount']:,.2f}")
print('Risk score:', example['risk_score'], '/ 100  ->', example['risk_tier'])
print('Recommended action:', example['recommended_action'])
print()
print('Explanation:')
print(example['explanation'])

## 6. Management narrative summary

The default (no API key required) template-based summary. The deployed app can optionally use
the Claude API for a more natural-language version of the same summary if the user supplies
their own key via Streamlit secrets -- entirely optional.

In [ ]:
print(agent.narrative_summary(scored))

## Conclusion

The hybrid rule + ML agent successfully surfaces the injected policy violations (over-limit
spend, missing receipts, self-approval, round-number amounts, weekend submissions, and
split/structured transactions) into the High/Medium risk tiers, while leaving the large
majority of normal, policy-compliant spend in the Low tier for auto-approval. This lets a
Finance Manager focus review time on the ~15-20% of transactions that actually need it,
instead of reviewing 100% of claims manually, while keeping every flag fully explainable for
audit purposes. See `README.md` in the repository root for deployment details and the live
Streamlit app link.